In [1]:
import sys, os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *

print("Fixed Income Library is loaded.")

Added to sys.path:/ /Users/tiffanyyan/Documents/GitHub/QuantBricks
Fixed Income Library is loaded.


### Create Build Method Collection

In [2]:
bm_list = []
# create yield curve build method
content_sofr = {
    "TARGET": "SOFR-1B",
    "OVERNIGHT INDEX FUTURE": "SOFR-FUTURE-3M",
    "OVERNIGHT INDEX SWAP": "USD-SOFR-OIS",
}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_INDEX", content_sofr))
# funding build method
content_sofr_funding = {
    "TARGET": "SOFR-1B-FLAT",
    "SPREAD ZERO RATE": "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD",
}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_sofr_funding))
# funding build method
content_bond_funding = {"TARGET": "USD-GOVT-BOND-FUNDING", "BOND FIXED": "USD-GOVT-BOND-FIXED"}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_FUNDING", content_bond_funding))
# create yield curve common build method
content = {"TARGET": "USD", "FUNDING PARAMETERS": "USD-FUNDING-PARAMETERS", "SOLVER": "BRENTQ"}
bm_list.append(qfCreateBuildMethod("YIELD_CURVE_COMMON", content))
build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

In [3]:
build_method_collection.display()

,Name,Value
0,YIELD_CURVE_INDEX,SOFR-1B
1,YIELD_CURVE_FUNDING,SOFR-1B-FLAT
2,YIELD_CURVE_FUNDING,USD-GOVT-BOND-FUNDING
3,YIELD_CURVE_COMMON,USD


### Create Data Collection

In [4]:
### ois futures
df_fut = pd.DataFrame(
    [
        ["2026-03-19x2026-06-18", 96.44],
        ["2026-06-18x2026-09-17", 96.70],
        ["2026-09-17x2026-12-10", 96.85],
        ["2026-12-10x2027-03-17", 96.90],
        ["2027-03-17x2027-06-16", 96.91],
        ["2027-06-16x2027-09-15", 96.89],
        ["2027-09-15x2027-12-15", 96.85],
        ["2027-12-15x2028-03-15", 96.81],
        ["2028-03-15x2028-06-21", 96.76],
        ["2028-06-21x2028-09-20", 96.71],
        ["2028-09-20x2028-12-20", 96.66],
        ["2028-12-20x2029-03-21", 96.61],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_fut = qfCreateData1D("OVERNIGHT INDEX FUTURE", "SOFR-FUTURE-3M", df_fut)

In [5]:
### ois swap
df_swap = pd.DataFrame(
    [
        ["4Y", 0.03358],
        ["5Y", 0.03422],
        ["6Y", 0.03491],
        ["7Y", 0.03560],
        ["8Y", 0.03624],
        ["9Y", 0.03685],
        ["10Y", 0.03742],
        ["12Y", 0.03849],
        ["15Y", 0.03974],
        ["20Y", 0.04087],
        ["25Y", 0.04110],
        ["30Y", 0.04089],
        ["35Y", 0.04044],
        ["40Y", 0.03996],
        ["50Y", 0.03887],
        ["60Y", 0.03772],
    ],
    columns=["axis1", "values"],
).set_index("axis1")
data_swap = qfCreateData1D("OVERNIGHT INDEX SWAP", "USD-SOFR-OIS", df_swap)

In [6]:
# spread zero rate
df_spread_zero_rate_rfr = pd.DataFrame(
    [["1Y", 0.0], ["5Y", 0.0], ["10Y", 0.0], ["20Y", 0.0], ["30Y", 0.0]],
    columns=["axis1", "values"],
).set_index("axis1")
data_szr_rfr = qfCreateData1D(
    "SPREAD ZERO RATE", "SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD", df_spread_zero_rate_rfr
)

In [7]:
### bond

# register bond reference
ct_5 = qfCreateBondSpecs(
    "CT5",
    parameters={
        "ISIN": "CT5",
        "BOND_CONVENTION": "USD-GOVT-BOND-FIXED",
        "ISSUE_DATE": "2026-02-17",
        "FIRST_ACCRUAL_DATE": "2026-02-15",
        "FIRST_COUPON_DATE": "2026-08-15",
        "MATURITY_DATE": "2031-02-15",
        "COUPON_RATE": 0.035,
        "REDEMPTION_PERCENTAGE": 1.0,
    },
)

ct_10 = qfCreateBondSpecs(
    "CT10",
    parameters={
        "ISIN": "CT10",
        "BOND_CONVENTION": "USD-GOVT-BOND-FIXED",
        "ISSUE_DATE": "2026-02-17",
        "FIRST_ACCRUAL_DATE": "2026-02-15",
        "FIRST_COUPON_DATE": "2026-08-15",
        "MATURITY_DATE": "2036-02-15",
        "COUPON_RATE": 0.04,
        "REDEMPTION_PERCENTAGE": 1.0,
    },
)

In [8]:
product_bond_5 = qfCreateProductFromDataConvention(
    "2026-02-27", "USD-GOVT-BOND-FIXED", ct_5.name, 0.03  # dirty price
)
product_bond_10 = qfCreateProductFromDataConvention(
    "2026-02-27", "USD-GOVT-BOND-FIXED", ct_10.name, 0.04  # dirty price
)

In [9]:
### funding parameter table
df_dg = pd.DataFrame(
    [
        ["Overnight Index Future", "SOFR-FUTURE-3M", "SOFR-1B-FLAT"],
        ["Overnight Index Swap", "USD-SOFR-OIS", "SOFR-1B-FLAT"],
    ],
    columns=["DATA TYPE", "DATA CONVENTION", "FUNDING IDENTIFIER"],
)
data_fpt = qfCreateDataGeneric("DATA GENERIC", "USD-FUNDING-PARAMETERS", df_dg)

In [10]:
# create market data
df_bond = pd.DataFrame(
    [
        [ct_10.name, 0.04],
        [ct_5.name, 0.042],
    ],
    columns=["axis1", "values"],
).set_index("axis1")

data_bond = qfCreateData1D("BOND FIXED", "USD-GOVT-BOND-FIXED", df_bond)

In [11]:
### pack up all data into data collection
data_collection = qfCreateDataCollection([data_szr_rfr, data_fut, data_swap, data_bond, data_fpt])

### Create Model Yield Curve

In [12]:
value_date = "2026-02-11"
yc_model = qfCreateModel(value_date, "YIELD_CURVE", data_collection, build_method_collection)
path = "serialized/yc_model_calibrated.pickle"
qfWriteModelObjectToFile(yc_model, path)

'DONE'

In [13]:
### create bond funding vp
fi_vp_bond = qfCreateValuationParameters(
    "FUNDING INDEX PARAMETER",
    {
        "Funding Index": "SOFR-1B-FLAT",
        "CURRENCIES": "USD",
        "UNDERLYING FUNDING INDEX": "USD-GOVT-BOND-FUNDING",
    },
)
vp_collection_bond = qfCreateValuationParametersCollection([fi_vp_bond])


settlement_date = "2028-02-11"
### create calibration product
calib_prod = qfCreateProductBond(
    name=ct_10.name, trade_date=settlement_date, buy_sell="LONG", trade=0
)

### value report
qfCreateValueReport(yc_model, calib_prod, vp_collection_bond, "pv")

[['USD', 95.70820723376373]]

In [14]:
risk = qfCreateValueReport(yc_model, calib_prod, vp_collection_bond, "firstorderrisk")
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(10)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-0.007159
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,-0.012058
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.000000
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.000000
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000
5,BOND FIXED,USD-GOVT-BOND-FIXED,CT5,,0.042,0.0001,0.019714
6,BOND FIXED,USD-GOVT-BOND-FIXED,CT10,,0.04,0.0001,-0.084593
7,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,-0.003346
8,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,-0.002399
9,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-0.002217


In [15]:
# Step 1: build bumped up model
bump_size = 1e-4
df_bond_bumped = df_bond.copy()
df_bond_bumped.loc[ct_10.name] += bump_size
pv_base = qfCreateValueReport(yc_model, calib_prod, vp_collection_bond, "pv")[0][1]

In [16]:
data_bond_bumped = qfCreateData1D("BOND FIXED", "USD-GOVT-BOND-FIXED", df_bond_bumped)
data_collection_bumped = qfCreateDataCollection(
    [data_szr_rfr, data_fut, data_swap, data_bond_bumped, data_fpt]
)
yc_model_bumped = qfCreateModel(
    value_date, "YIELD_CURVE", data_collection_bumped, build_method_collection
)

pv_bumped = qfCreateValueReport(yc_model_bumped, calib_prod, vp_collection_bond, "pv")[0][1]

In [17]:
# Step 3 : change of value due to change of rate by 1 bp
risk_bump_reval = pv_bumped - pv_base
print(f"Bump reval risk is {risk_bump_reval}.")
# Step 4: bump reval risk - analytic risk

Bump reval risk is -0.08456434083414877.


## Test Portfolio

In [18]:
bond1=qfCreateProductBond(
    name=ct_5.name, trade_date=settlement_date, buy_sell="LONG", trade=101
)
bond2=qfCreateProductBond(
    name=ct_10.name, trade_date=settlement_date, buy_sell="LONG", trade=102
)
portfolio= qfCreatePortfolio([bond1, bond2], weights=[1,1])

In [19]:
qfDisplayProduct(portfolio)

,Name,Value
0,Product Type,PRODUCT_PORTFOLIO
1,Product 0 Type,PRODUCT_BOND
2,Product 0 Weight,1
3,Product 1 Type,PRODUCT_BOND
4,Product 1 Weight,1


In [20]:
ValuationEngineProductRegistry().display_registry()

Key : ('YIELD_CURVE', 'PRODUCT_PORTFOLIO', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.valuation.valuation_engine_portfolio.ValuationEngineProductPortfolio'>.
Key : ('YIELD_CURVE', 'PRODUCT_BULLET_CASHFLOW', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.yield_curve.valuation_engine.ValuationEngineProductBulletCashflow'>.
Key : ('YIELD_CURVE', 'PRODUCT_FIXED_ACCRUED', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.yield_curve.valuation_engine.ValuationEngineProductFixedAccrued'>.
Key : ('YIELD_CURVE', 'PRODUCT_RFR_FUTURE', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.yield_curve.valuation_engine.ValuationEngineProductRfrFuture'>.
Key : ('YIELD_CURVE', 'PRODUCT_INTEREST_RATE_STREAM', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.yield_curve.valuation_engine.ValuationEngineInterestRateStream'>.
Key : ('YIELD_CURVE', 'PRODUCT_RFR_SWAP', 'ANALYTIC PARAMETER'), Value : <class 'fixedincomelib.yield_curve.valuation_engine.ValuationEngineProductRfrSwap'

In [21]:
# qfCreateValueReport(yc_model, bond1, vp_collection_bond, "pv")[0][1]
risk = qfCreateValueReport(yc_model, bond1, vp_collection_bond, "firstorderrisk")
df_risk_1 = risk.display()
df_risk_1.VALUES = df_risk_1.VALUES.round(10)
# display(df_risk)

In [22]:
risk = qfCreateValueReport(yc_model, bond2, vp_collection_bond, "firstorderrisk")
df_risk_2 = risk.display()
df_risk_2.VALUES = df_risk_2.VALUES.round(10)
# display(df_risk_2)

In [23]:
df_risk_agg = pd.merge(
    df_risk_1[["DATA_TYPE", "AXIS1", "VALUES"]],
    df_risk_2[["DATA_TYPE", "AXIS1", "VALUES"]],
    on=["DATA_TYPE", "AXIS1"],
    how="outer",
).fillna(0)
df_risk_agg["SUM"] = df_risk_agg.VALUES_x + df_risk_agg.VALUES_y
display(df_risk_agg)

,DATA_TYPE,AXIS1,VALUES_x,VALUES_y,SUM
0,BOND FIXED,CT10,0.000000,-0.084593,-0.084593
1,BOND FIXED,CT5,-0.026367,0.019714,-0.006653
2,OVERNIGHT INDEX FUTURE,2026-03-19x2026-06-18,0.000043,-0.000006,0.000037
3,OVERNIGHT INDEX FUTURE,2026-06-18x2026-09-17,0.000031,-0.000004,0.000027
4,OVERNIGHT INDEX FUTURE,2026-09-17x2026-12-10,0.000028,-0.000004,0.000025
5,OVERNIGHT INDEX FUTURE,2026-12-10x2027-03-17,0.000033,-0.000004,0.000028
6,OVERNIGHT INDEX FUTURE,2027-03-17x2027-06-16,0.000031,-0.000004,0.000027
7,OVERNIGHT INDEX FUTURE,2027-06-16x2027-09-15,0.000031,-0.000004,0.000027
8,OVERNIGHT INDEX FUTURE,2027-09-15x2027-12-15,0.000031,-0.000004,0.000027
9,OVERNIGHT INDEX FUTURE,2027-12-15x2028-03-15,0.000021,-0.000003,0.000018


In [24]:
risk = qfCreateValueReport(yc_model, portfolio, vp_collection_bond, "firstorderrisk")
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(10)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,0.000079
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,0.000133
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.000000
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.000000
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000
5,BOND FIXED,USD-GOVT-BOND-FIXED,CT5,,0.042,0.0001,-0.006653
6,BOND FIXED,USD-GOVT-BOND-FIXED,CT10,,0.04,0.0001,-0.084593
7,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,0.000037
8,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,0.000027
9,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,0.000025


In [25]:
compare_df = pd.merge(
    df_risk[["DATA_TYPE", "AXIS1", "VALUES"]],
    df_risk_agg[["DATA_TYPE", "AXIS1", "SUM"]],
    on=["DATA_TYPE", "AXIS1"],
    how="outer",
).fillna(0)
display(compare_df)

,DATA_TYPE,AXIS1,VALUES,SUM
0,BOND FIXED,CT10,-0.084593,-0.084593
1,BOND FIXED,CT5,-0.006653,-0.006653
2,OVERNIGHT INDEX FUTURE,2026-03-19x2026-06-18,0.000037,0.000037
3,OVERNIGHT INDEX FUTURE,2026-06-18x2026-09-17,0.000027,0.000027
4,OVERNIGHT INDEX FUTURE,2026-09-17x2026-12-10,0.000025,0.000025
5,OVERNIGHT INDEX FUTURE,2026-12-10x2027-03-17,0.000028,0.000028
6,OVERNIGHT INDEX FUTURE,2027-03-17x2027-06-16,0.000027,0.000027
7,OVERNIGHT INDEX FUTURE,2027-06-16x2027-09-15,0.000027,0.000027
8,OVERNIGHT INDEX FUTURE,2027-09-15x2027-12-15,0.000027,0.000027
9,OVERNIGHT INDEX FUTURE,2027-12-15x2028-03-15,0.000018,0.000018


### Re-Price Calibration Instruments

In [28]:
### everything is collateralised in RFR
fi_vp = qfCreateValuationParameters("FUNDING INDEX PARAMETER", {"Funding Index": "SOFR-1B-FLAT"})
vp_collection = qfCreateValuationParametersCollection([fi_vp])
### re-price each product
req = "parrateorspread"
dc_display = data_collection.display()
for data in data_collection:
    if data.data_shape != "DATAGENERIC" and data.data_type != "SPREAD ZERO RATE":
        for _, row in data.display().iterrows():
            axis, quote = row["axis1"], row["values"]
            this_product = qfCreateProductFromDataConvention(
                value_date, data.data_convention.name, axis, quote
            )
            par_rate = qfCreateValueReport(yc_model, this_product, vp_collection, req)
            if "FUTURE" in data.data_type:
                print(f"target is {quote:.2f}, and model implied is {100 - 100 * par_rate:.2f}.")
            else:
                print(f"target is {quote:.2%}, and model implied is {par_rate:.2%}.")
                pv = qfCreateValueReport(yc_model, this_product, vp_collection, "pv")
                print(f"Pv of the swap is {pv[0][1]:.2f}.")

target is 96.44, and model implied is 96.44.
target is 96.70, and model implied is 96.70.
target is 96.85, and model implied is 96.85.
target is 96.90, and model implied is 96.90.
target is 96.91, and model implied is 96.91.
target is 96.89, and model implied is 96.89.
target is 96.85, and model implied is 96.85.
target is 96.81, and model implied is 96.81.
target is 96.76, and model implied is 96.76.
target is 96.71, and model implied is 96.71.
target is 96.66, and model implied is 96.66.
target is 96.61, and model implied is 96.61.
target is 3.36%, and model implied is 3.36%.
Pv of the swap is -0.00.
target is 3.42%, and model implied is 3.42%.
Pv of the swap is -0.00.
target is 3.49%, and model implied is 3.49%.
Pv of the swap is -0.00.
target is 3.56%, and model implied is 3.56%.
Pv of the swap is -0.00.
target is 3.62%, and model implied is 3.62%.
Pv of the swap is -0.00.
target is 3.69%, and model implied is 3.69%.
Pv of the swap is -0.00.
target is 3.74%, and model implied is 3.

Exception: This product does not support par rate or spread calculation.

# Fixed Accrued test

In [ ]:
# parametric product
effecitve_date = "2026-03-26"
termination_date = "2026-05-26"
currency = "USD"
notional = 1e6
accrual_basis = "ACT/360"
business_day_convention = "F"
holiday_convention = "USGS"
pay_offset = "2D"
payment_date = qfAddPeriod(
    termination_date, pay_offset, business_day_convention, holiday_convention
)
prod_fixed_accrued = qfCreateProducFixedAccrued(
    effecitve_date, termination_date, currency, notional, accrual_basis
)

In [ ]:
display(qfDisplayProduct(prod_fixed_accrued))

,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000.0
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-03-26
5,Termination Date,2026-05-26
6,Accrual Basis,ACT/360
7,Payment Date,2026-05-26
8,Business Day Convention,F
9,Holiday Convention,USGS


In [ ]:
qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "pv")

[['USD', 167718.44750591737]]

In [ ]:
### test risk
risk = qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "firstorderrisk")
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(10)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-4.778827
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,5Y,,0.0,0.0001,0.000000
2,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,10Y,,0.0,0.0001,0.000000
3,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,20Y,,0.0,0.0001,0.000000
4,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,30Y,,0.0,0.0001,0.000000
5,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,-4.801987
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,-0.000000
7,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-0.000000
8,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,-0.000000
9,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2027-03-17x2027-06-16,,96.91,-0.01,-0.000000


In [ ]:
qfCreateValueReport(yc_model, prod_fixed_accrued, vp_collection, "cashflowsreport").display().T

,0
PRODUCT_TYPE,PRODUCT_FIXED_ACCRUED
VALUATION_ENGINE_TYPE,ValuationEngineProductFixedAccrued
LEG_ID,0
CASHFLOW_ID,0
PAY_OR_RECEIVE,1.0
NOTIONAL,1000000.0
PAY_DATE,"May 26th, 2026"
FORECASTED_AMOUNT,169444.444444
PV,167718.447506
DISCOUNG FACTOR,0.989814


In [ ]:
engine = ValuationEngineProductRegistry().get(
    ("YIELD_CURVE", "PRODUCT_FIXED_ACCRUED", "ANALYTIC PARAMETER")
)(
    model=yc_model,
    product=prod_fixed_accrued,
    valuation_parameters_collection=vp_collection,
    request=req,
)

In [ ]:
engine.calculate_value()

In [ ]:
pv_cash = engine.get_value_and_cash()
cf_report = engine.create_cash_flows_report()

In [ ]:
cf_report.display().T

,0
PRODUCT_TYPE,PRODUCT_FIXED_ACCRUED
VALUATION_ENGINE_TYPE,ValuationEngineProductFixedAccrued
LEG_ID,0
CASHFLOW_ID,0
PAY_OR_RECEIVE,1.0
NOTIONAL,1000000.0
PAY_DATE,"May 26th, 2026"
FORECASTED_AMOUNT,169444.444444
PV,167718.447506
DISCOUNG FACTOR,0.989814


# Bond 

In [ ]:
bond = qfCreateProductBond(
    isin="US9128284W34",
    settlement_date=Date("2026-02-24"),
    buy_sell="LONG",
)
bond.currency

In [ ]:
qfCreateValueReport(yc_model, bond, vp_collection, "pvdetailed").display()

,Currency,Type,Value
0,USD,PV,100.482855
1,USD,CASH,0.000000
